# **02 - Exploratory Data Analysis (EDA)**
---

## **Table of Content**
[Objective](#obj)

[Dataset Info](#data-info)

[Summary](#summary)

## <a id ='obj'></a>**Objective**
To uncover temporal patterns, seasonalities and feature correlations to inform effective feature engineering, data preprocessing, and model architecture selection for accurate time series forecasting.

## <a id ='data-info'></a> **Dataset Info**
* **Source:** UCI Machine Learning Repository
* **Title:** Appliances Energy Prediction
* **Time Range:** 11th January 2016 - 27th May 2016
* **Primary Target Variable:** `Appliances` (Watt-hour consumed)

## **Domain-Related Questions**
In this section, we'll define the set of questions we'd like to answer from this EDA.

> **Q1**: What are the dominant daily and weekly consumption cycles, and does downsampling from 10-minute intervals to 30-minute or 1-hour steps preserve key signals while reducing noise?

> **Q2**: Do energy habits stay the same from January to May, or does the changing season (winter to spring) shift baseline power usage?

> **Q3**: How strongly does the current electrical consumption depends on past information, and how far back does the influence persist?

> **Q4**: Which environmental variables correlate most strongly with appliance energy usage, and are these relationships immediate or lagged?

> **Q5**: What influences wild fluctuation in appliances electrical usage, and does log transformation stabilize the variance?



## <a id = 'summary'></a>**Summary**

## **Import Necessary Libraries**

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import math


# Add the root directory (one level up from notebooks/) to Python's path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.preprocess import preprocess_energy

## **Import Raw Dataset**

In [ ]:
PROJECT_ROOT = Path.cwd().parent

# Import csv file
energy = pd.read_csv(f'{PROJECT_ROOT}/data/raw/energydata_complete.csv')

In [ ]:
# Print samples
energy.head(3)

# **Preprocess Using Method**

In [ ]:
# Call preprocess_energy method
energy = preprocess_energy(energy)

# Print preprocessed samples
energy.head(3)

## **Initial Inspection**

We'll first inspect the dataset characteristics, such as the shape, data types and so on.

In [ ]:
print(f'Number of Rows: {energy.shape[0]}')
print(f'Number of Columns: {energy.shape[1]}')

print('\n\n---DATASET INFORMATION---')
print(energy.info())

In [ ]:
# Print statistics overview
energy.describe()

## **Question-Driven Analysis**

Every question in this section follows a standardized 4-step structure:

1. **Question**: The target question being investigated.

2. **Hypothesis**: Anticipated finding based on domain context or prior logic.

3. **Analysis**: Iterative plots paired directly with analytical notes.

4. **Summary and Takeaways**: Key insights translated into concrete feature engineering decision points.

5. **Decision**: Decisions taken from the conclusion drawn.

<br>

---

> ### **Q1**: *What are the dominant daily and weekly consumption cycles, and does downsampling from 10-minute intervals to lower frequency steps preserve key signals while reducing noise?*

#### **Hypothesis**
Daily consumption will **peak during daytime/evening hours** and **drop overnight** during sleep, with **higher overall volume on weekends** due to home occupancy. Downsampling to lower frequency steps  will **smooth out sensor noise**, but it may fail to preserving the core peaks and fluctuations required for modelling when the downsampling is done too aggressively.

#### **Analysis**

#### A Typical Daily Consumption Pattern
We'll aggregate the 4-months data by hour, calculate it's mean and plot it into a time-series plot.

In [ ]:
# Create canvas
fig, ax = plt.subplots(figsize=(12, 5))

# Plot the aggregated trend
sns.lineplot(
    data=energy, 
    x='hour', 
    y='Appliances', 
    color="orange", 
    marker='o', 
    linewidth=2,
    ax=ax,
    label = 'Appliances Energy',
    errorbar=('ci', False)
)

# Display settings
ax.set_xticks(range(0, 24))
ax.set_xticklabels([f"{h}:00" for h in range(24)], rotation=45)

plt.title("Appliance Energy Throughout Time of Day", fontsize=20)
plt.xlabel("Time of Day", fontsize=14)
plt.ylabel("Average Consumption", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.5)

# Show the graph
plt.tight_layout()
plt.show()

 #### Observations
> **```1. Lowest baseline overnight```**

> **```2. Consumption ramps up in the morning and hit local maxima in  the afternoon```**

> **```3. Consumption peaks in the evening (19:00 hrs)```**

> **```4. Absolute maxima at 18:00 before steadily declining through midnight```**
 


#### A Typical Weekly Consumption Pattern
We'll do the same thing again, but this time grouping by day of the week.

In [ ]:
# Create canvas
fig, ax = plt.subplots(figsize=(12, 5))

# Plot the aggregated trend
sns.lineplot(
    data=energy, 
    x='day', 
    y='Appliances', 
    color="orange", 
    marker='o', 
    linewidth=2,
    ax=ax,
    label = 'Appliances Energy',
    errorbar=('ci', False)
)

# Display settings
plt.title("Appliance Energy Throughout Day of the Week", fontsize=20)
plt.xlabel("Day", fontsize=14)
plt.ylabel("Average Consumption", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.5)

# Show the graph
plt.tight_layout()
plt.show()

#### Observations
> **```1. Highest consumption on Monday```**

> **```2. Plateau from Tuesday - Thursday```**

> **```3. Higher Friday/Saturday consumption```**

> **```4. Lower consumption on Sunday```**

#### Choosing A Sample Week

Because we're dealing with a massive amount of rows (over 19k rows), plotting all of them to see the effect of downsampling would be overwhelmingly cluttered. For that, we'll only use the **median 7 days** data.

In [ ]:
# Slice median week
mid = math.ceil(len(energy)/2)
med_week = energy.iloc[mid:mid + 1008]

#### Experiment: Downsampling Effect

We'll consider these downsampling step size:
* 20 minutes
* 30 minutes
* 60 minutes

In [ ]:
steps = ['20min', '30min', '1h']

fig, ax = plt.subplots(nrows=3, ncols=1, figsize=(12, 10), sharex=True)

for i, item in enumerate(steps):
    # 1. Plot raw data
    sns.lineplot(
        data=med_week,
        x=med_week.index, 
        y='Appliances', 
        label='Raw Appliances',
        ax=ax[i], 
        alpha=0.3
    )

    # 2. Resample 
    downsampled = med_week['Appliances'].resample(item).max()

    # 3. Plot downsampled reading
    sns.lineplot(
        x=downsampled.index,
        y=downsampled.values,
        ax=ax[i],
        label=f'Downsampled Appliances: 1 sample/{item}',
        color='red'
    )

    # Display settings
    ax[i].set_title(f"10-min vs {item} Steps Appliances Reading", fontsize=14)
    ax[i].set_ylabel("Appliances")
    ax[i].grid(True, linestyle="--", alpha=0.5)

ax[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

#### Observations

> **```1. Loss of information in the 1h step size```**

> **```2. Insignificant smoothing in 20 min step size```**

> **```3. 30 min step size is a balanced version```**

---
> ### **Q2**: *Do energy habits stay the same from January to May, or does the changing season (winter to spring) shift baseline power usage?*

In [ ]:
# Create canvas
fig, ax = plt.subplots(nrows=5, ncols=1, figsize=(10, 12), sharex=True)

months = ['January', 'February', 'March', 'April', 'May']

# Loop through each day and plot it on its respective subplot row
for i, month in enumerate(months):
    df = energy[energy['month'] == month]
    
    sns.lineplot(
        data=df, 
        x='hour', 
        y='Appliances', 
        linewidth=1.5,
        ax=ax[i],
        label=month,
        errorbar=('ci', False)
    )
    
    ax[i].set_ylabel("Average Consumption")
    ax[i].grid(True, linestyle="--", alpha=0.4)
    ax[i].set_title(f'{month} Usage Trend')
    ax[i].set_xticks([h for h in range (24)])
    ax[i].set_xticklabels([f'{h}:00' for h in range (24)])

for axs in ax.flat:
    axs.tick_params(labelbottom=True)
    axs.tick_params(axis='x', labelrotation=45)

plt.tight_layout()
plt.show()

---
> ### **Q3**: *How strongly does the current electrical consumption depends on past information, and how far back does the influence persist?*

---
> ### **Q4**: *Which environmental variables correlate most strongly with appliance energy usage, and are these relationships immediate or lagged?*

---
> ### **Q5**: *What influences wild fluctuation in appliances electrical usage, and does log transformation stabilize the variance?*